In [2]:
import pandas as pd

# Load CSV
df = pd.read_csv(r"..\Datasets\intermediate\bone_ppis_EOA_predictions.csv")

# # Select relevant columns
# df = df[["uidA", "uidB", "Predicted Classes", "Probability Score", "Regression Value"]]

# # Compute mean of 'Probability Score' and 'Regression Value'
# df["mean_prob_aff"] = df[["Probability Score", "Regression Value"]].mean(axis=1)

# # Save to new CSV
# df.to_csv(r"..\Datasets\intermediate\bone_ppis_EOA_predictions.csv", index=False)

# print("Processing complete. File saved as 'processed_file.csv'.")


In [3]:
print(df)

          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877496  P17612  Q9H1M4                  1           0.517073   
877497  P29474  Q8IV53                  1           0.517073   
877498  P12931  Q6ZSI9                  1           0.517073   
877499  P28482  Q9H496                  1           0.507317   
877500  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  
0               0.401441       0.459257  
1               0.401442       0.461696  
2               0.401441       0.459257  
3               0.401441       0.459257  
4               0.401

In [4]:
import pandas as pd

# Load CSV
string_ds = pd.read_csv(r"..\Datasets\reference\stringdb_ppis_curated.csv")
print(string_ds)

                      string_A              string_B  score    uidA    uidB  \
0         9606.ENSP00000000233  9606.ENSP00000356607    173  P26437  B7Z7B1   
1         9606.ENSP00000000233  9606.ENSP00000427567    154  P26437  Q9C0D6   
2         9606.ENSP00000000233  9606.ENSP00000253413    151  P26437  A8MUE4   
3         9606.ENSP00000000233  9606.ENSP00000493357    471  P26437  A8K8P0   
4         9606.ENSP00000000233  9606.ENSP00000324127    201  P26437  A6NFQ4   
...                        ...                   ...    ...     ...     ...   
13715399  9606.ENSP00000501317  9606.ENSP00000475489    195  Q2KHR2  A1L486   
13715400  9606.ENSP00000501317  9606.ENSP00000370447    158  Q2KHR2  A0JNS5   
13715401  9606.ENSP00000501317  9606.ENSP00000312272    226  Q2KHR2  A0PK16   
13715402  9606.ENSP00000501317  9606.ENSP00000402092    169  Q2KHR2  B4DY20   
13715403  9606.ENSP00000501317  9606.ENSP00000404074    251  Q2KHR2  B4E1W6   

          score_norm  
0           0.027091  
1    

In [8]:
import re
# Strip whitespaces
df["uidA"] = df["uidA"].astype(str).str.strip()
df["uidB"] = df["uidB"].astype(str).str.strip()
string_ds["uidA"] = string_ds["uidA"].astype(str).str.strip()
string_ds["uidB"] = string_ds["uidB"].astype(str).str.strip()

# Define regex pattern for UniProt IDs with '-Number' suffix
uniprot_suffix_pattern = r"^[A-Z0-9]+-\d+$"

# Count occurrences of IDs with suffix
df_suffix_count = df["uidA"].str.contains(uniprot_suffix_pattern).sum() + df["uidB"].str.contains(uniprot_suffix_pattern).sum()
string_ds_suffix_count = string_ds["uidA"].str.contains(uniprot_suffix_pattern).sum() + string_ds["uidB"].str.contains(uniprot_suffix_pattern).sum()

print(f"UniProt IDs with '-Number' suffix in df: {df_suffix_count}")
print(f"UniProt IDs with '-Number' suffix in string_ds: {string_ds_suffix_count}")

# Function to remove the "-Number" suffix from UniProt IDs
def remove_suffix(uid):
    return re.sub(r"-\d+$", "", uid)

# Create new columns without the suffix for alternative matching
df["uidA_clean"] = df["uidA"].apply(remove_suffix)
df["uidB_clean"] = df["uidB"].apply(remove_suffix)
string_ds["uidA_clean"] = string_ds["uidA"].apply(remove_suffix)
string_ds["uidB_clean"] = string_ds["uidB"].apply(remove_suffix)

# Create a unified key for both original and cleaned versions
df["pair"] = df["uidA"] + "-" + df["uidB"]
df["pair_reverse"] = df["uidB"] + "-" + df["uidA"]
df["pair_clean"] = df["uidA_clean"] + "-" + df["uidB_clean"]
df["pair_reverse_clean"] = df["uidB_clean"] + "-" + df["uidA_clean"]

string_ds["pair"] = string_ds["uidA"] + "-" + string_ds["uidB"]
string_ds["pair_clean"] = string_ds["uidA_clean"] + "-" + string_ds["uidB_clean"]

# Merge on both original and cleaned pair names
merged = df.merge(string_ds, on="pair", how="left", suffixes=("", "_str"))
merged_reverse = df.merge(string_ds, left_on="pair_reverse", right_on="pair", how="left", suffixes=("", "_str"))
merged_clean = df.merge(string_ds, on="pair_clean", how="left", suffixes=("", "_clean"))
merged_reverse_clean = df.merge(string_ds, left_on="pair_reverse_clean", right_on="pair_clean", how="left", suffixes=("", "_clean"))

# Combine results to check if the pair exists in any direction, with or without the suffix
merged["stringdb_check"] = merged["score_norm"].combine_first(merged_reverse["score_norm"])
merged["stringdb_check"] = merged["stringdb_check"].combine_first(merged_clean["score_norm"])
merged["stringdb_check"] = merged["stringdb_check"].combine_first(merged_reverse_clean["score_norm"])
merged["stringdb_check"] = merged["stringdb_check"].notna().astype(int)  # Convert existence to 1 or 0

# Keep only required columns
merged = merged[["uidA", "uidB", "Predicted Classes", "Probability Score", "Regression Value", "mean_prob_aff", "stringdb_check", "score_norm"]]


print(merged)


UniProt IDs with '-Number' suffix in df: 0
UniProt IDs with '-Number' suffix in string_ds: 2518
          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877500  P17612  Q9H1M4                  1           0.517073   
877501  P29474  Q8IV53                  1           0.517073   
877502  P12931  Q6ZSI9                  1           0.517073   
877503  P28482  Q9H496                  1           0.507317   
877504  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  
0               0.401441       0.459257               0         N

In [9]:
print(merged['stringdb_check'].value_counts())
print(merged[merged['stringdb_check']==1])

stringdb_check
0    869670
1      7835
Name: count, dtype: int64
          uidA    uidB  Predicted Classes  Probability Score  \
142     Q04206  Q8TED1                  0           0.507317   
352     P05412  Q16576                  1           0.517073   
657     P04198  Q13207                  0           0.507317   
717     P03372  P30559                  0           0.541463   
1047    P04637  P05154                  0           0.536585   
...        ...     ...                ...                ...   
876901  P17612  P30990                  0           0.536585   
876974  P18075  P42702                  1           0.517073   
877070  O75367  P05412                  0           0.536585   
877249  P04637  Q969G5                  1           0.517073   
877369  P12643  P60508                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  
142             0.401305       0.454311               1    0.163722  
352             0.401779  

In [22]:
print (merged)



          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877500  P17612  Q9H1M4                  1           0.517073   
877501  P29474  Q8IV53                  1           0.517073   
877502  P12931  Q6ZSI9                  1           0.517073   
877503  P28482  Q9H496                  1           0.507317   
877504  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  
0               0.401441       0.459257               0         NaN  
1               0.401442       0.461696               0         NaN  
2               0.401

In [23]:
merged2=merged
# Identify duplicate rows in merged dataset
duplicates_merged = merged2[merged2.duplicated(subset=["uidA", "uidB"], keep=False)]

# Print duplicate rows (if any)
if not duplicates_merged.empty:
    print("Duplicate rows found in merged dataset:")
    print(duplicates_merged)

# Remove duplicates
merged_cleaned_2 = merged2.drop_duplicates(subset=["uidA", "uidB"], keep="first")
print("\n------------------------------------------------------------------------------------\n")

Duplicate rows found in merged dataset:
              uidA    uidB  Predicted Classes  Probability Score  \
53720       P04637  Q9ULZ0                  1           0.517073   
53721       P04637  Q9ULZ0                  1           0.517073   
288974      O15389  Q04206                  1           0.517073   
288975      O15389  Q04206                  1           0.517073   
569522      P04637  Q16385                  1           0.517073   
569523      P04637  Q16385                  1           0.517073   
855327  A0A087WV53  P18075                  1           0.517073   
855328  A0A087WV53  P18075                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  
53720           0.401441       0.459257               1    0.000000  
53721           0.401441       0.459257               1    0.000000  
288974          0.401940       0.459507               1    0.021201  
288975          0.401940       0.459507               1    0.021201

In [25]:
print (merged_cleaned_2)
merged_cleaned_2.to_csv(r"..\Datasets\intermediate\bone_ppis_EOA_predictions_stringdb_check.csv", index=False)

          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877500  P17612  Q9H1M4                  1           0.517073   
877501  P29474  Q8IV53                  1           0.517073   
877502  P12931  Q6ZSI9                  1           0.517073   
877503  P28482  Q9H496                  1           0.507317   
877504  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  
0               0.401441       0.459257               0         NaN  
1               0.401442       0.461696               0         NaN  
2               0.401

In [27]:

# Load the irefindex dataset (only contains uidA and uidB)
irefindex_ds = pd.read_csv(r"..\Datasets\reference\irefindex_v3.csv")

print(irefindex_ds)

           uidA        uidB  \
0        L7RTG7      Q06124   
1        Q6ZW61  A0A024R687   
2        Q6ZW61  A0A024R687   
3        L7RTG7      Q06124   
4        Q9Y2I6      Q8N157   
...         ...         ...   
59922    Q14289      Q9Y6R4   
59923    Q14289      Q9Y6R4   
59924    P01106      O95782   
59925  P53367-2      P53365   
59926    P30679    Q9BTE6-2   

                                                  method  \
0      psi-mi:"MI:0012"(bioluminescence resonance ene...   
1      psi-mi:"MI:2222"(inference by socio-affinity s...   
2         psi-mi:"MI:0676"(tandem affinity purification)   
3      psi-mi:"MI:0729"(luminescence based mammalian ...   
4      psi-mi:"MI:2222"(inference by socio-affinity s...   
...                                                  ...   
59922                psi-mi:"MI:0428"(imaging technique)   
59923                psi-mi:"MI:0428"(imaging technique)   
59924  psi-mi:"MI:0006"(anti bait coimmunoprecipitation)   
59925   psi-mi:"MI:0007"(an

In [28]:

# Function to remove the "-Number" suffix from UniProt IDs
def remove_suffix(uid):
    return re.sub(r"-\d+$", "", str(uid))  # Ensure it's a string before applying regex

# Strip whitespaces and remove suffixes
for df in [merged_cleaned_2, irefindex_ds]:
    df["uidA"] = df["uidA"].astype(str).str.strip().apply(remove_suffix)
    df["uidB"] = df["uidB"].astype(str).str.strip().apply(remove_suffix)

# Create a pair identifier for both normal and reversed orders
merged_cleaned_2["pair"] = merged_cleaned_2["uidA"] + "-" + merged_cleaned_2["uidB"]
merged_cleaned_2["pair_reverse"] = merged_cleaned_2["uidB"] + "-" + merged_cleaned_2["uidA"]

irefindex_ds["pair"] = irefindex_ds["uidA"] + "-" + irefindex_ds["uidB"]

# Merge on both original and reversed pair names to check existence
merged_iref = merged_cleaned_2.merge(irefindex_ds, on="pair", how="left", indicator=True)
merged_reverse_iref = merged_cleaned_2.merge(irefindex_ds, left_on="pair_reverse", right_on="pair", how="left", indicator=True)

# Assign 1 if found in either direction, else 0
merged_iref["irefindex_check"] = (merged_iref["_merge"] == "both") | (merged_reverse_iref["_merge"] == "both")
merged_iref["irefindex_check"] = merged_iref["irefindex_check"].astype(int)

# Keep only required columns
merged_iref = merged_iref.drop(columns=["pair", "pair_reverse", "_merge"])


print(merged_iref)

<ipython-input-28-5f2594fd0de1>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["uidA"] = df["uidA"].astype(str).str.strip().apply(remove_suffix)
<ipython-input-28-5f2594fd0de1>:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["uidB"] = df["uidB"].astype(str).str.strip().apply(remove_suffix)
<ipython-input-28-5f2594fd0de1>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu

        uidA_x  uidB_x  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877832  P17612  Q9H1M4                  1           0.517073   
877833  P29474  Q8IV53                  1           0.517073   
877834  P12931  Q6ZSI9                  1           0.517073   
877835  P28482  Q9H496                  1           0.507317   
877836  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm uidA_y  \
0               0.401441       0.459257               0         NaN    NaN   
1               0.401442       0.461696               0         NaN    NaN 

In [30]:
merged_iref.rename(columns={"uidA_x": "uidA", "uidB_x": "uidB", "uidA_y": "uidA_irefindex", "uidB_y": "uidB_irefindex"}, inplace=True)

# Identify duplicate rows based on uidA and uidB
duplicates_merged = merged_iref[merged_iref.duplicated(subset=["uidA", "uidB"], keep=False)]

# Print duplicate rows (if any)
if not duplicates_merged.empty:
    print("Duplicate rows found in final dataset:")
    print(duplicates_merged)

# Remove duplicates, keeping the first occurrence
merged_final = merged_iref.drop_duplicates(subset=["uidA", "uidB"], keep="first")


Duplicate rows found in final dataset:
          uidA    uidB  Predicted Classes  Probability Score  \
1488    O75582  Q04206                  1           0.517073   
1489    O75582  Q04206                  1           0.517073   
1766    P28482  Q12913                  1           0.517073   
1767    P28482  Q12913                  1           0.517073   
1768    P28482  Q12913                  1           0.517073   
...        ...     ...                ...                ...   
859893  P01106  Q13105                  1           0.512195   
859894  P01106  Q13105                  1           0.512195   
859895  P01106  Q13105                  1           0.512195   
875198  P12931  Q16825                  1           0.517073   
875199  P12931  Q16825                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  \
1488            0.401441       0.459257               0         NaN   
1489            0.401441       0.459257           

In [31]:
print(merged_final)

          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877832  P17612  Q9H1M4                  1           0.517073   
877833  P29474  Q8IV53                  1           0.517073   
877834  P12931  Q6ZSI9                  1           0.517073   
877835  P28482  Q9H496                  1           0.507317   
877836  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  \
0               0.401441       0.459257               0         NaN   
1               0.401442       0.461696               0         NaN   
2               0.

In [35]:
print(merged_final["stringdb_check"].value_counts()) 
#print(merged_final[merged_final["irefindex_check"]==1]) 
merged_final.to_csv(r"..\Datasets\intermediate\bone_ppis_EOA_predictions_stringdb_AND_irefindex_check.csv", index=False)


stringdb_check
0    869670
1      7831
Name: count, dtype: int64


In [47]:
import requests
import numpy as np

# Load the main dataset
merged_final = pd.read_csv(r"..\Datasets\intermediate\bone_ppis_EOA_predictions_stringdb_AND_irefindex_check.csv")
print("Original dataset shape:", merged_final.shape)

# Load the mapping dataset (uid -> Gene Name)
uid_to_gene = pd.read_csv(r"..\Datasets\reference\uid_to_Gene_Names.csv")

# Convert mapping dataset into a dictionary for fast lookup
uid_gene_dict = dict(zip(uid_to_gene['uid'], uid_to_gene['ID']))  # 'ID' column contains gene names

# Map GeneA and GeneB using only the local dataset
merged_final['GeneA'] = merged_final['uidA'].map(uid_gene_dict)
merged_final['GeneB'] = merged_final['uidB'].map(uid_gene_dict)

# Clean up trailing semicolons
merged_final['GeneA'] = merged_final['GeneA'].astype(str).str.rstrip(';')
merged_final['GeneB'] = merged_final['GeneB'].astype(str).str.rstrip(';')


# Print the final dataset
print("Updated dataset shape:", merged_final.shape)
print(merged_final.head())


Original dataset shape: (877501, 14)
Updated dataset shape: (877501, 16)
     uidA    uidB  Predicted Classes  Probability Score  Regression Value  \
0  P12524  Q92538                  1           0.517073          0.401441   
1  P31749  Q5T7N2                  0           0.521951          0.401442   
2  P16442  P29474                  1           0.517073          0.401441   
3  P0DPD6  P12931                  1           0.517073          0.401441   
4  P05997  Q9GZQ8                  0           0.502439          0.401456   

   mean_prob_aff  stringdb_check  score_norm uidA_irefindex uidB_irefindex  \
0       0.459257               0         NaN            NaN            NaN   
1       0.461696               0         NaN            NaN            NaN   
2       0.459257               0         NaN            NaN            NaN   
3       0.459257               0         NaN            NaN            NaN   
4       0.451948               0         NaN            NaN            NaN

In [48]:
# Save the updated dataset
output_path = r"..\Datasets\intermediate\bone_ppis_EOA_predictions_stringdb_AND_irefindex_check_AND_GeneNames.csv"
merged_final.to_csv(output_path, index=False)
print (merged_final)

          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877496  P17612  Q9H1M4                  1           0.517073   
877497  P29474  Q8IV53                  1           0.517073   
877498  P12931  Q6ZSI9                  1           0.517073   
877499  P28482  Q9H496                  1           0.507317   
877500  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  \
0               0.401441       0.459257               0         NaN   
1               0.401442       0.461696               0         NaN   
2               0.

In [61]:
bone_list = pd.read_excel(r"..\Datasets\reference\bone_proteins_04.xlsx")  # Dataset to be mapped

# Create a set of uids from bone_list (which contains the gene names corresponding to the uids)
uids_set = set(bone_list['uid'])  # 'uid' is the column in bone_list that contains UniProt IDs

# Create OP_A based on the existence of uidA in the uids_set (check if uidA is in bone_list)
merged_final['OP_A'] = merged_final['uidA'].apply(lambda x: 1 if x in uids_set else 0)

# Create OP_B based on the existence of uidB in the uids_set (check if uidB is in bone_list)
merged_final['OP_B'] = merged_final['uidB'].apply(lambda x: 1 if x in uids_set else 0)

# Create OP_check column that is 1 if either OP_A or OP_B is 1, otherwise 0
merged_final['OP_check'] = merged_final[['OP_A', 'OP_B']].max(axis=1)

# Print the resulting DataFrame and the value counts for the OP_check column

print(merged_final['OP_check'].value_counts())

OP_check
1    877501
Name: count, dtype: int64


In [62]:
print(merged_final)
output_path = r"..\Datasets\intermediate\bone_ppis_EOA_predictions_stringdb_AND_irefindex_check_AND_GeneNames_AND_OPcheck.csv"
merged_final.to_csv(output_path, index=False)

          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P31749  Q5T7N2                  0           0.521951   
2       P16442  P29474                  1           0.517073   
3       P0DPD6  P12931                  1           0.517073   
4       P05997  Q9GZQ8                  0           0.502439   
...        ...     ...                ...                ...   
877496  P17612  Q9H1M4                  1           0.517073   
877497  P29474  Q8IV53                  1           0.517073   
877498  P12931  Q6ZSI9                  1           0.517073   
877499  P28482  Q9H496                  1           0.507317   
877500  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  \
0               0.401441       0.459257               0         NaN   
1               0.401442       0.461696               0         NaN   
2               0.

In [63]:
positive_ds=merged_final[merged_final['Predicted Classes']==1]

positive_ds.to_csv(r"..\Datasets\intermediate\bone_ppis_EOA_positive_fully_mapped.csv", index=False)

In [1]:
import pandas as pd
positive_ds=pd.read_csv(r"..\Datasets\intermediate\bone_ppis_EOA_positive_fully_mapped.csv")
print(positive_ds)


          uidA    uidB  Predicted Classes  Probability Score  \
0       P12524  Q92538                  1           0.517073   
1       P16442  P29474                  1           0.517073   
2       P0DPD6  P12931                  1           0.517073   
3       O75525  P35354                  1           0.517073   
4       P10451  Q6DHY5                  1           0.517073   
...        ...     ...                ...                ...   
479017  P17612  Q9H1M4                  1           0.517073   
479018  P29474  Q8IV53                  1           0.517073   
479019  P12931  Q6ZSI9                  1           0.517073   
479020  P28482  Q9H496                  1           0.507317   
479021  P04629  Q9UQF2                  1           0.517073   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  \
0               0.401441       0.459257               0         NaN   
1               0.401441       0.459257               0         NaN   
2               0.

In [2]:
print(positive_ds['Probability Score'].describe())


count    479022.000000
mean          0.516263
std           0.004065
min           0.502439
25%           0.517073
50%           0.517073
75%           0.517073
max           0.970732
Name: Probability Score, dtype: float64


In [9]:
value_counts = positive_ds['mean_prob_aff'].value_counts(normalize=True) * 100

# Get top 5 values by percentage
top_5 = value_counts.head(5)
print(top_5)
print(positive_ds['mean_prob_aff'].describe())


mean_prob_aff
0.459257    0.000418
0.459257    0.000418
0.459257    0.000418
0.459257    0.000418
0.459257    0.000418
Name: proportion, dtype: float64
count    479022.000000
mean          0.458876
std           0.002052
min           0.451872
25%           0.459257
50%           0.459257
75%           0.459290
max           0.692132
Name: mean_prob_aff, dtype: float64


In [3]:
# Analyze Positive Ds

print("Full Positive DS shape\n", positive_ds.shape, "\n")
print("\nIrefindex Value Counts:", positive_ds['irefindex_check'].value_counts(), "\n")
print("\nStringDB Value Counts:", positive_ds['stringdb_check'].value_counts(), "\n")
print("\nOP check:", positive_ds['OP_check'].value_counts(), "\n")

positive_ds_filter_prob= positive_ds[positive_ds['Probability Score']>0.517074]

print("\nProbability (>0,517074) Filter Positive DS shape\n", positive_ds_filter_prob.shape, "\n")
print("\nIrefindex Value Counts:", positive_ds_filter_prob['irefindex_check'].value_counts(), "\n")
print("\nStringDB Value Counts:", positive_ds_filter_prob['stringdb_check'].value_counts(), "\n")
print("\nOP check:", positive_ds_filter_prob['OP_check'].value_counts(), "\n")

positive_ds_filter_mean= positive_ds[positive_ds['mean_prob_aff']> 0.459257]

print("\nMean Prob-Aff (> 0.459257) Filter Positive DS shape\n", positive_ds_filter_mean.shape, "\n")
print("\nIrefindex Value Counts:", positive_ds_filter_mean['irefindex_check'].value_counts(), "\n")
print("\nStringDB Value Counts:", positive_ds_filter_mean['stringdb_check'].value_counts(), "\n")
print("\nOP check:", positive_ds_filter_mean['OP_check'].value_counts(), "\n")

positive_ds_filter_both = positive_ds[
    (positive_ds['Probability Score'] > 0.517074) & 
    (positive_ds['mean_prob_aff'] > 0.459257)
]
print("\nProbability (>0,517074) & Mean Prob-Aff (> 0.459257) Filter  Positive DS shape\n", positive_ds_filter_both.shape, "\n")
print("\nIrefindex Value Counts:", positive_ds_filter_both['irefindex_check'].value_counts(), "\n")
print("\nStringDB Value Counts:", positive_ds_filter_both['stringdb_check'].value_counts(), "\n")
print("\nOP check:", positive_ds_filter_both['OP_check'].value_counts(), "\n")


Full Positive DS shape
 (479022, 19) 


Irefindex Value Counts: irefindex_check
0    478498
1       524
Name: count, dtype: int64 


StringDB Value Counts: stringdb_check
0    475153
1      3869
Name: count, dtype: int64 


OP check: OP_check
1    479022
Name: count, dtype: int64 


Probability (>0,517074) Filter Positive DS shape
 (1079, 19) 


Irefindex Value Counts: irefindex_check
0    1072
1       7
Name: count, dtype: int64 


StringDB Value Counts: stringdb_check
0    1066
1      13
Name: count, dtype: int64 


OP check: OP_check
1    1079
Name: count, dtype: int64 


Mean Prob-Aff (> 0.459257) Filter Positive DS shape
 (420489, 19) 


Irefindex Value Counts: irefindex_check
0    420049
1       440
Name: count, dtype: int64 


StringDB Value Counts: stringdb_check
0    416964
1      3525
Name: count, dtype: int64 


OP check: OP_check
1    420489
Name: count, dtype: int64 


Probability (>0,517074) & Mean Prob-Aff (> 0.459257) Filter  Positive DS shape
 (1079, 19) 


Irefindex V

In [4]:
# CHOOSE THE PROBABILITY + MEAN PROB-AFF FILTER (BOTH FILTER)
#Probability (>0,517074) & Mean Prob-Aff (> 0.459257) Filter

chosen_filter_ds= positive_ds_filter_both

In [17]:
#RE-ADD ANY ROWS  CONTAINING OP-OP interactions

# Load the list of UIDs from Excel
bone_list = pd.read_excel(r"..\Datasets\reference\bone_proteins_04.xlsx")   # Read only 'uid' column
uid_list = set(bone_list["uid"])

# Step 2: Identify rows where both uidA and uidB are in the UID list
uids_in_list = chosen_filter_ds[
    (chosen_filter_ds['uidA'].isin(uid_list)) & (chosen_filter_ds['uidB'].isin(uid_list))
]

# Step 3: Find the rows that were **not included** in the initial filtering
recovered_rows = uids_in_list[~uids_in_list.index.isin(chosen_filter_ds.index)]

# Step 4: Combine original filtered rows with the recovered ones
final_filtered_dataset = pd.concat([chosen_filter_ds, recovered_rows]).drop_duplicates()

# Display recovered rows
print("Rows that were re-added:")
print(recovered_rows)


Rows that were re-added:
Empty DataFrame
Columns: [uidA, uidB, Predicted Classes, Probability Score, Regression Value, mean_prob_aff, stringdb_check, score_norm, uidA_irefindex, uidB_irefindex, method, Host_organism_taxid, numParticipants, irefindex_check, GeneA, GeneB, OP_A, OP_B, OP_check]
Index: []


In [18]:
print(final_filtered_dataset)

          uidA    uidB  Predicted Classes  Probability Score  \
257     P01106  P25789                  1           0.551220   
533     P01258  Q14050                  1           0.551220   
1886    O43895  P29474                  1           0.551220   
2164    P01701  Q9BQB4                  1           0.521951   
2311    P04198  Q9UJK0                  1           0.551220   
...        ...     ...                ...                ...   
478422  P43235  Q8TBE0                  1           0.551220   
478469  Q86U38  Q9UNN8                  1           0.551220   
478529  P02452  P18887                  1           0.551220   
478589  P05997  Q9NU63                  1           0.551220   
478730  O75665  P12524                  1           0.521951   

        Regression Value  mean_prob_aff  stringdb_check  score_norm  \
257             0.401633       0.476426               0         NaN   
533             0.401633       0.476426               0         NaN   
1886            0.

In [ ]:

from itertools import combinations
# Step 2: Find unique uids where OP_A or OP_B is 0
uids_with_op_0 = set(final_filtered_dataset.loc[final_filtered_dataset['OP_A'] == 0, 'uidA']).union(set(final_filtered_dataset.loc[final_filtered_dataset['OP_B'] == 0, 'uidB']))
print("Unique UIDs with OP_A or OP_B as 0:", len(uids_with_op_0))

# Save unique UIDs to a CSV with a single column 'uid'
uid_df = pd.DataFrame({'uid': list(uids_with_op_0)})
uid_df.to_csv(r"..\Datasets\intermediate\BOTH_FILTER_OP_interactor_ids.csv", index=False)
print("Saved unique UIDs to CSV.")

# Step 3: Generate all possible interactions (A-B and B-A are the same)
interaction_pairs = list(combinations(uids_with_op_0, 2))
interaction_df = pd.DataFrame(interaction_pairs, columns=['uidA', 'uidB'])

# Save the new dataset
interaction_df.to_csv(r"..\Datasets\intermediate\BOTH_FILTER_OP_interactor_combs.csv", index=False)
print("Generated interactions dataset with", len(interaction_df), "rows.")

Unique UIDs with OP_A or OP_B as 0: 995
Saved unique UIDs to CSV.
